# 06_query_transformation: Multi-Query, Decomposition, and HyDE

This notebook demonstrates three query rewriting techniques (Multi-Query Expansion, Query Decomposition, and Hypothetical Document Embeddings (HyDE)) using OpenAI LLMs to optimize retrieval coverage.

### Query Transformation Mechanics
1. **Multi-Query Expansion**: Solves lexical mismatch. An LLM expands query $q$ into alternative phrasings $\{q^{(1)}, q^{(2)}, q^{(3)}\}$, searching the database for all candidates to increase recall.
2. **Query Decomposition**: Solves compound/multi-hop search problems. Given a complex query $q_{\text{compound}}$, the LLM breaks it down into sub-queries $\{q^{(1)}, q^{(2)}\}$, retrieving contexts for each and joining them before synthesis.
3. **HyDE (Hypothetical Document Embeddings)**: Prompts an LLM to generate a hypothetical answer $d_{\text{hyp}}$. The embedding $\phi(d_{\text{hyp}})$ is used as the lookup vector, matching document-to-document geometry rather than query-to-document geometry.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(dotenv_path=r"d:\\Study\\Prep\\.env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Multi-Query Expansion
multi_query_prompt = ChatPromptTemplate.from_template(
    "Generate three alternative search queries for: '{query}'\nOutput queries only, one per line."
)
multi_query_chain = multi_query_prompt | llm | StrOutputParser()
alternatives = multi_query_chain.invoke({"query": "Compare iPhone 15 battery vs Samsung S24"})
print("Transformed Queries (Multi-Query):\n", alternatives)

Transformed Queries (Multi-Query):
 - 'iPhone 15 battery life comparison with Samsung Galaxy S24'
- 'iPhone 15 vs Samsung S24 battery performance review'
- 'Battery comparison: iPhone 15 and Samsung S24'


In [3]:
# Query Decomposition
decomp_prompt = ChatPromptTemplate.from_template(
    "Break down this compound question into two distinct search queries: '{query}'\nOutput queries only, one per line."
)
decomp_chain = decomp_prompt | llm | StrOutputParser()
subqueries = decomp_chain.invoke({"query": "Compare iPhone 15 battery life and find its price"})
print("Decomposed Queries:\n", subqueries)

Decomposed Queries:
 Compare iPhone 15 battery life  
Find iPhone 15 price


In [4]:
# HyDE (Hypothetical Document Embeddings)
hyde_prompt = ChatPromptTemplate.from_template(
    "Write a hypothetical short paragraph answer for: '{query}'"
)
hyde_chain = hyde_prompt | llm | StrOutputParser()
hypothetical_answer = hyde_chain.invoke({"query": "What is semantic chunking?"})
print("HyDE Hypothetical Answer:\n", hypothetical_answer)

HyDE Hypothetical Answer:
 Semantic chunking is a cognitive strategy that involves breaking down complex information into smaller, meaningful units or "chunks" to enhance understanding and retention. By grouping related concepts or ideas together, individuals can more easily process and recall information. For example, when learning a new language, instead of memorizing individual words, a learner might focus on phrases or sentences that convey complete thoughts. This method leverages the brain's natural tendency to recognize patterns and relationships, making it a powerful tool for effective learning and memory enhancement.


### Output Explanation & Verification

#### Executed Results:
- **Multi-Query Alternative phrasings**:
  - *'iPhone 15 battery life comparison with Samsung Galaxy S24'*
  - *'iPhone 15 vs Samsung S24 battery performance review'*
  - *'Battery comparison: iPhone 15 and Samsung S24'*
- **Decomposed Queries** for *'Compare iPhone 15 battery life and find its price'*:
  1. *'Compare iPhone 15 battery life'*
  2. *'Find iPhone 15 price'*
- **HyDE Generated Hypothetical Answer**: Generates a rich explanatory paragraph about semantic chunking being a cognitive grouping strategy.

#### Production Insights:
- **Multi-Query** improves search recall by addressing vocabulary gaps, but increases LLM API count and search latency ($O(K)$ queries).
- **Decomposition** is crucial for multi-hop databases where target facts reside in separate, non-overlapping tables or documents.
- **HyDE** shifts similarity matching from query-document to document-document vector alignment, which works well in low-data regimes but can fail if the model generates highly incorrect hypothetical assertions.